In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.preprocessing import build_pipeline

# Week 2 — Pipeline, SMOTE, and Model Comparison

Builds on `src/preprocessing.py`. Compares Logistic Regression with/without SMOTE,
then Logistic Regression vs Random Forest on AUC-ROC and business cost.

In [4]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

col_names = [
    "status", "duration", "credit_history", "purpose", "amount",
    "savings", "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "number_credits", "job",
    "people_liable", "telephone", "foreign_worker", "credit_risk"
]

df = pd.read_csv(url, sep=" ", header=None, names=col_names)

In [5]:
# Decode Statlog categorical columns
status_map = {
    'A11': '< 0 DM', 'A12': '0–200 DM', 'A13': '>= 200 DM', 'A14': 'no account'
}
credit_history_map = {
    'A30': 'no credits', 'A31': 'all paid', 'A32': 'existing paid',
    'A33': 'delay in past', 'A34': 'critical account'
}
purpose_map = {
    'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture', 'A43': 'TV/radio',
    'A44': 'appliances', 'A45': 'repairs', 'A46': 'education', 'A47': 'vacation',
    'A48': 'retraining', 'A49': 'business', 'A410': 'other'
}
savings_map = {
    'A61': '< 100 DM', 'A62': '100–500 DM', 'A63': '500–1000 DM',
    'A64': '>= 1000 DM', 'A65': 'unknown/none'
}
employment_map = {
    'A71': 'unemployed', 'A72': '< 1 yr', 'A73': '1–4 yrs',
    'A74': '4–7 yrs', 'A75': '>= 7 yrs'
}
personal_status_map = {
    'A91': 'male divorced', 'A92': 'female divorced/married',
    'A93': 'male single', 'A94': 'male married', 'A95': 'female single'
}
other_debtors_map = {'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'}
property_map = {
    'A121': 'real estate', 'A122': 'savings/insurance',
    'A123': 'car/other', 'A124': 'unknown/none'
}
other_installment_map = {'A141': 'bank', 'A142': 'stores', 'A143': 'none'}
housing_map = {'A151': 'rent', 'A152': 'own', 'A153': 'free'}
job_map = {
    'A171': 'unskilled non-resident', 'A172': 'unskilled resident',
    'A173': 'skilled', 'A174': 'highly skilled'
}
telephone_map = {'A191': 'none', 'A192': 'yes'}
foreign_worker_map = {'A201': 'yes', 'A202': 'no'}

df_decoded = df.copy()
df_decoded['status'] = df['status'].map(status_map)
df_decoded['credit_history'] = df['credit_history'].map(credit_history_map)
df_decoded['purpose'] = df['purpose'].map(purpose_map)
df_decoded['savings'] = df['savings'].map(savings_map)
df_decoded['employment_duration'] = df['employment_duration'].map(employment_map)
df_decoded['personal_status_sex'] = df['personal_status_sex'].map(personal_status_map)
df_decoded['other_debtors'] = df['other_debtors'].map(other_debtors_map)
df_decoded['property'] = df['property'].map(property_map)
df_decoded['other_installment_plans'] = df['other_installment_plans'].map(other_installment_map)
df_decoded['housing'] = df['housing'].map(housing_map)
df_decoded['job'] = df['job'].map(job_map)
df_decoded['telephone'] = df['telephone'].map(telephone_map)
df_decoded['foreign_worker'] = df['foreign_worker'].map(foreign_worker_map)

# Make target binary: 0 = good, 1 = bad
df_decoded['credit_risk'] = df['credit_risk'].map({1: 0, 2: 1})

df_decoded.head()

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,< 0 DM,6,critical account,TV/radio,1169,unknown/none,>= 7 yrs,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,0
1,0–200 DM,48,existing paid,TV/radio,5951,< 100 DM,1–4 yrs,2,female divorced/married,none,...,real estate,22,none,own,1,skilled,1,none,yes,1
2,no account,12,critical account,education,2096,< 100 DM,4–7 yrs,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,0
3,< 0 DM,42,existing paid,furniture,7882,< 100 DM,4–7 yrs,2,male single,guarantor,...,savings/insurance,45,none,free,1,skilled,2,none,yes,0
4,< 0 DM,24,delay in past,car (new),4870,< 100 DM,1–4 yrs,3,male single,none,...,unknown/none,53,none,free,2,skilled,2,none,yes,1


## Train/Test Split

In [6]:
X = df_decoded.drop(columns=["credit_risk"])
y = df_decoded["credit_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

## Evaluation Helper

In [7]:
def evaluate(pipeline, X_test, y_test, label):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    print(f"--- {label} ---")
    print(classification_report(y_test, y_pred))

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cost = (fn * 5) + (fp * 1)
    print(f"False Negatives: {fn}, False Positives: {fp}")
    print(f"Business cost: {cost}")

    auc = roc_auc_score(y_test, y_proba)
    print(f"AUC-ROC: {auc:.3f}\n")

    return cost, auc

## SMOTE Comparison (Logistic Regression)

In [8]:
pipeline_baseline = build_pipeline(LogisticRegression(max_iter=1000, random_state=7), use_smote=False)
pipeline_baseline.fit(X_train, y_train)
cost_baseline, auc_baseline = evaluate(pipeline_baseline, X_test, y_test, "Without SMOTE")

pipeline_smote = build_pipeline(LogisticRegression(max_iter=1000, random_state=7), use_smote=True)
pipeline_smote.fit(X_train, y_train)
cost_smote, auc_smote = evaluate(pipeline_smote, X_test, y_test, "With SMOTE")

--- Without SMOTE ---
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       140
           1       0.72      0.57      0.64        60

    accuracy                           0.81       200
   macro avg       0.78      0.74      0.75       200
weighted avg       0.80      0.81      0.80       200

False Negatives: 26, False Positives: 13
Business cost: 143
AUC-ROC: 0.829

--- With SMOTE ---
              precision    recall  f1-score   support

           0       0.88      0.73      0.80       140
           1       0.55      0.77      0.64        60

    accuracy                           0.74       200
   macro avg       0.71      0.75      0.72       200
weighted avg       0.78      0.74      0.75       200

False Negatives: 14, False Positives: 38
Business cost: 108
AUC-ROC: 0.824



### Why SMOTE despite the accuracy drop

Adding SMOTE reduced overall accuracy but reduced the total business cost. This is expected
and desirable: accuracy treats all errors equally, but our cost matrix does not.

A false negative (approving an applicant who defaults) risks losing the loan principal
itself. A false positive (rejecting a good applicant) only costs the opportunity cost of
the interest we would have earned — a smaller, forgone-profit loss rather than a real
capital loss. Because principal loss is categorically worse than lost interest income,
the cost matrix weights false negatives 5x higher, and SMOTE's recall gain on the
minority class (bad credit) directly targets that weighted cost — even though it comes
at the expense of raw accuracy.

## Model Comparison: Logistic Regression vs Random Forest (SMOTE applied to both)

In [9]:
lr_pipeline = build_pipeline(LogisticRegression(max_iter=1000, random_state=7), use_smote=True)
lr_pipeline.fit(X_train, y_train)
cost_lr, auc_lr = evaluate(lr_pipeline, X_test, y_test, "Logistic Regression + SMOTE")

rf_pipeline = build_pipeline(RandomForestClassifier(random_state=7), use_smote=True)
rf_pipeline.fit(X_train, y_train)
cost_rf, auc_rf = evaluate(rf_pipeline, X_test, y_test, "Random Forest + SMOTE")

--- Logistic Regression + SMOTE ---
              precision    recall  f1-score   support

           0       0.88      0.73      0.80       140
           1       0.55      0.77      0.64        60

    accuracy                           0.74       200
   macro avg       0.71      0.75      0.72       200
weighted avg       0.78      0.74      0.75       200

False Negatives: 14, False Positives: 38
Business cost: 108
AUC-ROC: 0.824

--- Random Forest + SMOTE ---
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       140
           1       0.70      0.43      0.54        60

    accuracy                           0.78       200
   macro avg       0.75      0.68      0.69       200
weighted avg       0.76      0.78      0.76       200

False Negatives: 34, False Positives: 11
Business cost: 181
AUC-ROC: 0.816



In [10]:
import mlflow
import mlflow.sklearn

def evaluate_and_log(pipeline, X_test, y_test, label, params: dict):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    cost = (fn * 5) + (fp * 1)
    auc = roc_auc_score(y_test, y_proba)

    print(f"--- {label} ---")
    print(classification_report(y_test, y_pred))
    print(f"False Negatives: {fn}, False Positives: {fp}")
    print(f"Business cost: {cost}")
    print(f"AUC-ROC: {auc:.3f}\n")

    with mlflow.start_run(run_name=label):
        mlflow.log_params(params)
        mlflow.log_metric("business_cost", cost)
        mlflow.log_metric("auc_roc", auc)
        mlflow.log_metric("false_negatives", fn)
        mlflow.log_metric("false_positives", fp)
        mlflow.sklearn.log_model(pipeline, "model")

    return cost, auc

/Users/ahmedali/Projects/github/credit-risk-pipeline/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from xgboost import XGBClassifier

lr_pipeline = build_pipeline(LogisticRegression(max_iter=1000, random_state=7), use_smote=True)
lr_pipeline.fit(X_train, y_train)
cost_lr, auc_lr = evaluate_and_log(
    lr_pipeline, X_test, y_test, "Logistic Regression + SMOTE",
    params={"model": "LogisticRegression", "use_smote": True}
)

rf_pipeline = build_pipeline(RandomForestClassifier(random_state=7), use_smote=True)
rf_pipeline.fit(X_train, y_train)
cost_rf, auc_rf = evaluate_and_log(
    rf_pipeline, X_test, y_test, "Random Forest + SMOTE",
    params={"model": "RandomForestClassifier", "use_smote": True}
)

xgb_pipeline = build_pipeline(XGBClassifier(random_state=7, eval_metric="logloss"), use_smote=True)
xgb_pipeline.fit(X_train, y_train)
cost_xgb, auc_xgb = evaluate_and_log(
    xgb_pipeline, X_test, y_test, "XGBoost + SMOTE",
    params={"model": "XGBClassifier", "use_smote": True}
)

--- Logistic Regression + SMOTE ---
              precision    recall  f1-score   support

           0       0.88      0.73      0.80       140
           1       0.55      0.77      0.64        60

    accuracy                           0.74       200
   macro avg       0.71      0.75      0.72       200
weighted avg       0.78      0.74      0.75       200

False Negatives: 14, False Positives: 38
Business cost: 108
AUC-ROC: 0.824



2026/07/10 20:23:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/10 20:23:21 INFO mlflow.store.db.utils: Updating database tables
2026/07/10 20:23:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 20:23:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/10 20:23:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 20:23:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execu

--- Random Forest + SMOTE ---
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       140
           1       0.70      0.43      0.54        60

    accuracy                           0.78       200
   macro avg       0.75      0.68      0.69       200
weighted avg       0.76      0.78      0.76       200

False Negatives: 34, False Positives: 11
Business cost: 181
AUC-ROC: 0.816



2026/07/10 20:23:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 20:23:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- XGBoost + SMOTE ---
              precision    recall  f1-score   support

           0       0.82      0.86      0.84       140
           1       0.63      0.55      0.59        60

    accuracy                           0.77       200
   macro avg       0.73      0.71      0.71       200
weighted avg       0.76      0.77      0.76       200

False Negatives: 27, False Positives: 19
Business cost: 154
AUC-ROC: 0.788

